In [40]:
## Gradient Boosting Regressor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings(action='ignore')
%matplotlib inline

In [41]:
df = pd.read_csv('cardekho_imputated.csv')
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [42]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [43]:
df.drop(columns=['Unnamed: 0','car_name','brand'], axis=1, inplace=True)

In [44]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [45]:
len(df['model'].unique())

120

In [46]:
## Find numerical, categrrical, discrete, continuous feature

num_feature = [features for features in df.columns if df[features].dtype != 'O']
print('No. of numerical features ', len(num_feature))

cat_feature = [features for features in df.columns if df[features].dtype == 'O']
print('No. of categorical feature ', len(cat_feature))

dis_feature = [features for features in num_feature if len(df[features].unique()) <= 25]
print('No. of discrete features ', len(dis_feature))

con_feature = [features for features in num_feature if features not in dis_feature]
print('No. of categorical features ', len(cat_feature))

No. of numerical features  7
No. of categorical feature  4
No. of discrete features  2
No. of categorical features  4


In [47]:
## independent and dependent feature

X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

In [48]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [49]:
## Feature encoding And scalling

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [50]:
len(df['seller_type'].unique()), len(df['transmission_type'].unique()), len(df['fuel_type'].unique())

(3, 2, 5)

In [51]:
## create column transformer with 3 types of transformer

num_feature = X.select_dtypes(exclude='object').columns
cat_feature = ['seller_type', 'transmission_type', 'fuel_type']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ('StandardScaler', numerical_transformer, num_feature),
        ('OneHotEncoder', categorical_transformer, cat_feature)
    ],remainder='passthrough'
)

In [52]:
preprocessor

,transformers,"[('StandardScaler', ...), ('OneHotEncoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [53]:
X = preprocessor.fit_transform(X)

In [54]:
## split data
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.33,random_state=42)
X_train.shape, X_test.shape

((10325, 17), (5086, 17))

In [55]:
## Model Training and Selection

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [56]:
## function to evaluate model

def model_evaluate(true, predicted):
    mse = mean_squared_error(true, predicted)
    r2 = r2_score(true, predicted)
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mse)
    return mse, r2, mae, rmse

In [58]:
models = {
    'Linear Regression' : LinearRegression(),
    'AdaBoost Regressor' : AdaBoostRegressor(),
    'GradientBoosting Regressor' : GradientBoostingRegressor(),
    'DecisionTree Regressor' : DecisionTreeRegressor(),
    'Ridge' : Ridge(),
    'Lasso' : Lasso(),
    'KNeighbors Regressor' : KNeighborsRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    ## make prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mse, train_rmse, train_mae, train_r2 = model_evaluate(y_train, y_train_pred)

    test_mse, test_rmse, test_mae, test_r2 = model_evaluate(y_test, y_test_pred)

    ## print model performance
    print(list(models.keys())[i])
    print('----------Train Model Performance-------')
    print('mse: {:.2f}'.format(train_mse))
    print('rmse: {:.2f}'.format(train_rmse))
    print('mae: {:.2f}'.format(train_mae))
    print('r2-score: {:.2f}'.format(train_r2))

    print('-----------Test Model Performance--------')
    print('mse: {:.2f}'.format(test_mse))
    print('rmse: {:.2f}'.format(test_rmse))
    print('mae: {:.2f}'.format(test_mae))
    print('r2-score: {:.2f}'.format(test_r2))

    print('='*35)
    print('\n')

Linear Regression
----------Train Model Performance-------
mse: 317614523214.21
rmse: 0.62
mae: 269089.26
r2-score: 563573.00
-----------Test Model Performance--------
mse: 253401740454.38
rmse: 0.66
mae: 281003.41
r2-score: 503390.25


AdaBoost Regressor
----------Train Model Performance-------
mse: 152552707918.43
rmse: 0.82
mae: 272457.42
r2-score: 390579.96
-----------Test Model Performance--------
mse: 190252821235.12
rmse: 0.74
mae: 290569.47
r2-score: 436179.80


GradientBoosting Regressor
----------Train Model Performance-------
mse: 38267079611.53
rmse: 0.95
mae: 110084.02
r2-score: 195619.73
-----------Test Model Performance--------
mse: 69235921417.52
rmse: 0.91
mae: 125451.52
r2-score: 263127.20


DecisionTree Regressor
----------Train Model Performance-------
mse: 393543904.76
rmse: 1.00
mae: 4500.42
r2-score: 19837.94
-----------Test Model Performance--------
mse: 91203293905.56
rmse: 0.88
mae: 129966.44
r2-score: 301998.83


Ridge
----------Train Model Performance-------

In [60]:
knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}
Gd_params = {"loss": ['squared_error','huber','absolute_error'],
             "criterion": ['friedman_mse','squared_error','mse'],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500],
              "max_depth": [5, 8, 15, None, 10],
              "learning_rate": [0.1, 0.01, 0.02, 0.03]}

In [61]:
randomcv_model = [
    ('KNN', KNeighborsRegressor(), knn_params),
    ('GB', GradientBoostingRegressor(), Gd_params)
]

In [62]:
from sklearn.model_selection import RandomizedSearchCV
model_param = {}

for name,model,params in randomcv_model:
    random = RandomizedSearchCV(
        estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1
    )
    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

    for best_param in model_param:
        print(f'----------- Best Model {best_param} ---------')
        print(model_param[best_param])

Fitting 3 folds for each of 6 candidates, totalling 18 fits
----------- Best Model KNN ---------
{'n_neighbors': 2}
Fitting 3 folds for each of 100 candidates, totalling 300 fits
----------- Best Model KNN ---------
{'n_neighbors': 2}
----------- Best Model GB ---------
{'n_estimators': 500, 'min_samples_split': 20, 'max_depth': 5, 'loss': 'squared_error', 'learning_rate': 0.1, 'criterion': 'friedman_mse'}


In [63]:
models = {
    'GradientBoosting Regressor' : GradientBoostingRegressor(n_estimators=500, min_samples_split=20, max_depth=5, loss='squared_error',
                                                              learning_rate=0.1, criterion='friedman_mse'),
    'KNeighbors Regressor' : KNeighborsRegressor(n_neighbors=2)
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    ## make prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mse, train_rmse, train_mae, train_r2 = model_evaluate(y_train, y_train_pred)

    test_mse, test_rmse, test_mae, test_r2 = model_evaluate(y_test, y_test_pred)

    ## print model performance
    print(list(models.keys())[i])
    print('----------Train Model Performance-------')
    print('mse: {:.2f}'.format(train_mse))
    print('rmse: {:.2f}'.format(train_rmse))
    print('mae: {:.2f}'.format(train_mae))
    print('r2-score: {:.2f}'.format(train_r2))

    print('-----------Test Model Performance--------')
    print('mse: {:.2f}'.format(test_mse))
    print('rmse: {:.2f}'.format(test_rmse))
    print('mae: {:.2f}'.format(test_mae))
    print('r2-score: {:.2f}'.format(test_r2))

    print('='*35)
    print('\n')

GradientBoosting Regressor
----------Train Model Performance-------
mse: 8513086470.61
rmse: 0.99
mae: 64168.38
r2-score: 92266.39
-----------Test Model Performance--------
mse: 44069489964.81
rmse: 0.94
mae: 96993.22
r2-score: 209927.34


KNeighbors Regressor
----------Train Model Performance-------
mse: 52606447118.64
rmse: 0.94
mae: 66820.53
r2-score: 229360.95
-----------Test Model Performance--------
mse: 85988619817.27
rmse: 0.88
mae: 119100.50
r2-score: 293238.16


